# Feature Adoption Analysis

## Project Overview
Analyze user-level product usage data to study adoption rates of key features, segment adoption by user type and tenure, and identify which features correlate most with engagement and retention.

## Learning Objectives
- Calculate feature adoption rates from event-level usage data
- Segment adoption by user type, plan tier, and tenure cohort
- Identify features that drive overall engagement
- Visualize adoption funnels and feature usage distributions

## Problem Statement
A SaaS product team wants to understand which features users actually adopt after signing up. Some features may have low discovery rates despite being high-value, while others may be widely tried but quickly abandoned.

## Why This Project Matters
Feature adoption analysis informs product roadmaps. Features with low adoption but high retention impact deserve better onboarding. Features with high adoption but low impact may be consuming engineering resources without payoff.

## Dataset Overview
Synthetic SaaS product usage data: ~3K users, 8 key features, usage events over 12 months. Includes user plan tier, signup cohort, and engagement metrics.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- Patterns inspired by typical SaaS product analytics
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_users = 3000

features = ['Dashboard', 'Reports', 'Integrations', 'API Access',
            'Custom Alerts', 'Team Collaboration', 'Export CSV', 'Advanced Filters']

# Feature base adoption probabilities
feat_base_prob = {
    'Dashboard': 0.92, 'Reports': 0.70, 'Export CSV': 0.60,
    'Advanced Filters': 0.50, 'Custom Alerts': 0.35,
    'Team Collaboration': 0.30, 'Integrations': 0.25, 'API Access': 0.15,
}

plans = ['Free', 'Pro', 'Enterprise']
plan_weights = [0.45, 0.35, 0.20]
plan_multiplier = {'Free': 0.7, 'Pro': 1.0, 'Enterprise': 1.3}

cohorts = ['2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2']
cohort_weights = [0.12, 0.14, 0.16, 0.18, 0.20, 0.20]

rows = []
for uid in range(n_users):
    plan = np.random.choice(plans, p=plan_weights)
    cohort = np.random.choice(cohorts, p=cohort_weights)
    mult = plan_multiplier[plan]

    adopted = {}
    total_usage = 0
    for feat in features:
        prob = min(feat_base_prob[feat] * mult, 0.98)
        used = np.random.random() < prob
        adopted[feat] = used
        if used:
            # Usage intensity: number of times used in the period
            usage_count = max(1, int(np.random.exponential(scale=15 if feat == 'Dashboard' else 8)))
            adopted[f'{feat}_usage'] = usage_count
            total_usage += usage_count
        else:
            adopted[f'{feat}_usage'] = 0

    # Engagement = total feature usage events
    # Retained = still active (correlated with feature breadth)
    n_adopted = sum(1 for f in features if adopted[f])
    retained = np.random.random() < min(0.3 + n_adopted * 0.08, 0.95)

    row = {'UserID': f'U{uid:05d}', 'Plan': plan, 'Cohort': cohort,
           'FeaturesAdopted': n_adopted, 'TotalUsage': total_usage, 'Retained': retained}
    for feat in features:
        row[f'Used_{feat.replace(" ", "_")}'] = adopted[feat]
        row[f'Usage_{feat.replace(" ", "_")}'] = adopted[f'{feat}_usage']
    rows.append(row)

df = pd.DataFrame(rows)
feat_cols = [f'Used_{f.replace(" ", "_")}' for f in features]
usage_cols = [f'Usage_{f.replace(" ", "_")}' for f in features]
print(f'Dataset shape: {df.shape}')
print(f'Plans: {df["Plan"].value_counts().to_dict()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Unique users: {df["UserID"].nunique()}')
print(f'Retention rate: {df["Retained"].mean():.1%}')
print(f'Avg features adopted: {df["FeaturesAdopted"].mean():.1f}')
print(f'\nFeatures adopted range: {df["FeaturesAdopted"].min()} - {df["FeaturesAdopted"].max()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall adoption rate per feature
adoption_rates = df[feat_cols].mean().sort_values(ascending=True)
adoption_rates.index = [c.replace('Used_', '').replace('_', ' ') for c in adoption_rates.index]
adoption_rates.plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Feature Adoption Rate')
axes[0,0].set_xlabel('Adoption Rate')
axes[0,0].set_xlim(0, 1)

# Distribution of features adopted per user
df['FeaturesAdopted'].hist(ax=axes[0,1], bins=range(0, 10), edgecolor='black', alpha=0.7)
axes[0,1].set_title('Features Adopted per User')
axes[0,1].set_xlabel('# Features')
axes[0,1].set_ylabel('Users')

# Adoption by plan
plan_adoption = df.groupby('Plan')[feat_cols].mean().T
plan_adoption.index = [c.replace('Used_', '').replace('_', ' ') for c in plan_adoption.index]
plan_adoption.plot.barh(ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('Adoption Rate by Plan')
axes[1,0].set_xlabel('Adoption Rate')
axes[1,0].legend(title='Plan')

# Retention by features adopted
ret_by_feat = df.groupby('FeaturesAdopted')['Retained'].mean()
ret_by_feat.plot(ax=axes[1,1], marker='o', color='green')
axes[1,1].set_title('Retention Rate vs Features Adopted')
axes[1,1].set_xlabel('# Features Adopted')
axes[1,1].set_ylabel('Retention Rate')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Target Analysis — Adoption Depth

In [ ]:
print('Adoption depth summary:')
print(f'  0-2 features: {(df["FeaturesAdopted"] <= 2).sum()} users ({(df["FeaturesAdopted"] <= 2).mean():.1%})')
print(f'  3-5 features: {((df["FeaturesAdopted"] >= 3) & (df["FeaturesAdopted"] <= 5)).sum()} users')
print(f'  6-8 features: {(df["FeaturesAdopted"] >= 6).sum()} users ({(df["FeaturesAdopted"] >= 6).mean():.1%})')
print()
print('Retention by adoption depth:')
for label, mask in [('Low (0-2)', df['FeaturesAdopted'] <= 2),
                     ('Medium (3-5)', (df['FeaturesAdopted'] >= 3) & (df['FeaturesAdopted'] <= 5)),
                     ('High (6-8)', df['FeaturesAdopted'] >= 6)]:
    print(f'  {label}: {df.loc[mask, "Retained"].mean():.1%} retention')

## Feature-Level Adoption by Cohort

In [ ]:
cohort_adoption = df.groupby('Cohort')[feat_cols].mean()
cohort_adoption.columns = [c.replace('Used_', '').replace('_', ' ') for c in cohort_adoption.columns]

plt.figure(figsize=(12, 6))
sns.heatmap(cohort_adoption.T, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Feature Adoption Rate by Signup Cohort')
plt.ylabel('Feature')
plt.xlabel('Cohort')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cohort_adoption.png'), dpi=100, bbox_inches='tight')
plt.show()

## Feature Usage Intensity

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
usage_data = df[usage_cols].copy()
usage_data.columns = [c.replace('Usage_', '').replace('_', ' ') for c in usage_data.columns]
# Only non-zero usage
usage_melted = usage_data.melt(var_name='Feature', value_name='UsageCount')
usage_melted = usage_melted[usage_melted['UsageCount'] > 0]

sns.boxplot(data=usage_melted, x='Feature', y='UsageCount', ax=ax)
ax.set_title('Usage Intensity by Feature (Adopters Only)')
ax.set_ylabel('Usage Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'usage_intensity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Features Correlated with Retention

In [ ]:
retention_corr = pd.DataFrame({
    'Feature': features,
    'Adoption_Retention_Corr': [df[f'Used_{f.replace(" ", "_")}'].corr(df['Retained'].astype(int)) for f in features],
    'Usage_Retention_Corr': [df[f'Usage_{f.replace(" ", "_")}'].corr(df['Retained'].astype(int)) for f in features],
}).sort_values('Adoption_Retention_Corr', ascending=False)

print('Correlation with Retention:')
print(retention_corr.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(retention_corr))
ax.bar([i - 0.2 for i in x], retention_corr['Adoption_Retention_Corr'], width=0.4, label='Adoption', edgecolor='black')
ax.bar([i + 0.2 for i in x], retention_corr['Usage_Retention_Corr'], width=0.4, label='Usage Intensity', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(retention_corr['Feature'], rotation=45, ha='right')
ax.set_ylabel('Correlation with Retention')
ax.set_title('Feature Impact on Retention')
ax.legend()
ax.axhline(y=0, color='grey', linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'retention_correlation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Adoption Funnel — Ordered by Typical Discovery Sequence

In [ ]:
discovery_order = ['Dashboard', 'Reports', 'Export CSV', 'Advanced Filters',
                    'Custom Alerts', 'Team Collaboration', 'Integrations', 'API Access']
funnel_rates = [df[f'Used_{f.replace(" ", "_")}'].mean() for f in discovery_order]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(discovery_order[::-1], funnel_rates[::-1], edgecolor='black', color=plt.cm.Blues(np.linspace(0.3, 0.9, 8)))
ax.set_xlabel('Adoption Rate')
ax.set_title('Feature Adoption Funnel')
for bar, rate in zip(bars, funnel_rates[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{rate:.0%}',
            va='center', fontsize=10)
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'adoption_funnel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Dashboard** is nearly universally adopted — it's the default landing experience
- **API Access** and **Integrations** have the lowest adoption but may correlate strongly with retention (power-user features)
- Enterprise users adopt significantly more features than Free users, validating the value proposition of paid tiers
- Users who adopt 6+ features retain at a much higher rate — the "aha moment" appears around 3-4 features
- **Custom Alerts** and **Team Collaboration** are underadopted relative to their retention correlation — candidates for better onboarding

## Limitations
- Synthetic data with pre-programmed adoption probabilities
- No temporal sequence of feature discovery (when each feature was first used)
- Binary adoption doesn't capture depth (used once vs. daily)
- Retention is a single binary; real retention is measured over time windows

## How to Improve This Project
- Use real product analytics data (e.g., Mixpanel/Amplitude exports)
- Add time-to-first-use for each feature to understand discovery speed
- Segment by user persona or company size for B2B products
- Build a predictive model: given feature adoption in week 1, predict retention at month 3
- Track feature adoption trends across product releases

## Production Considerations
- Instrument in-app feature usage tracking (event-level, not just session-level)
- Build automated adoption dashboards segmented by plan, cohort, and persona
- Set up adoption rate alerts for new feature launches
- Use adoption data to trigger in-app guidance for underadopted high-value features

## Common Mistakes
- Conflating feature adoption (ever used) with feature engagement (regularly used)
- Assuming all features are equally important for all user segments
- Not controlling for plan tier when comparing adoption rates
- Measuring adoption counts without connecting to business outcomes (retention, revenue)

## Mini Challenge / Exercises
1. Which feature has the highest gap between Free and Enterprise adoption rates?
2. Create a "power user" segment (adopted 7+ features). How do they differ from average users?
3. For each feature, calculate the retention lift: retention rate of adopters vs non-adopters.
4. If you could improve onboarding for one feature, which would have the biggest retention impact?

## Final Summary / Key Takeaways
- Feature adoption analysis connects product usage to business outcomes
- Not all features matter equally — focus onboarding efforts on high-impact, low-adoption features
- Feature breadth (number of features adopted) is a strong predictor of retention
- Plan tier is a major adoption driver — ensure Free users can discover value to drive upgrades
- Adoption funnels reveal natural discovery sequences and friction points